# Week 19: Experiment B, final runs, clinical metrics, robustness (milestone M8)

| Session | Topic | Time |
|---|---|---|
| Day 1 | Select on validation, then run the test set **once** | 2 h (+ GPU time on Colab) |
| Day 2 | Clinically meaningful metrics: thresholds, sensitivity/specificity, CIs | 3 h |
| Day 3 | A robustness probe: what happens when the images change? | 2.5 h |
| Day 4 | Table B, figures, and the Experiment B results write-up | 2 h |
| Review | Spaced recall | 0.5 h |

> ⚠️ Whatever the numbers, these models are **not clinically validated**: single-source data, tiny images, unknown patient overlap, and no prospective evaluation. Say so in every write-up.

### Milestone M8
- [ ] Validation runs for all 4 models (3 seeds each); the model is selected **on validation only**.
- [ ] Test results (once) with bootstrap 95% CIs, sensitivity and specificity at a threshold chosen on validation, and ECE.
- [ ] A robustness curve, plus `results/table_B.md`, `report/figures/fig_B_*.png`, and `report/results_B.md`.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: Validation first, then test once (2 h)

1. Run all four imaging configs on **val**. Run `resnet_finetune` on Colab or Kaggle (see Week 16) and copy its JSON back into `results/imaging/`.
2. **Before looking at test:** write in `PROTOCOL.md`, under a new heading `## Model selection (filled in before test)`, which model you selected and why.
3. Run every model on **test** once.

### Exercise 19.1: Add a `--split` override to `medlab/imaging/run.py`
`python -m medlab.imaging.run --config configs/imaging_small_cnn.yaml --split test` should behave as if the config said `split: test`, so you don't need duplicate config files.

In [ ]:
# Add the --split flag, then run the next cell (it trains everything on CPU; set SKIP_FINETUNE = True if you ran it on Colab).

In [ ]:
import subprocess, sys, json
import numpy as np
SKIP_FINETUNE = False
models = ["logreg", "small_cnn", "resnet_frozen"] + ([] if SKIP_FINETUNE else ["resnet_finetune"])
for split in ["val", "test"]:
    for m in models:
        out = subprocess.run([sys.executable, "-m", "medlab.imaging.run", "--config", f"configs/imaging_{m}.yaml", "--split", split],
                             cwd=CAP, capture_output=True, text=True)
        assert out.returncode == 0, out.stderr[-1500:]
res = {(m, s): json.load(open(CAP / "results" / "imaging" / f"{m}_{s}.json")) for m in models for s in ["val", "test"]}
for m in models:
    print(f"{m:16s} val AUROC {res[(m, 'val')]['mean']:.3f} ± {res[(m, 'val')]['std']:.3f}")
selected = max(models, key=lambda m: res[(m, "val")]["mean"])
print("selected on validation:", selected)
print("✅ 19.1 passed")

---
# Day 2: Clinical metrics (3 h)

AUROC summarises ranking quality, but a deployed classifier needs a **threshold**, and then it has a sensitivity (the share of pneumonia cases caught) and a specificity (the share of normal cases correctly cleared). **Choose the threshold on validation**; choosing it on test is leakage (Week 8).

Two common rules:
- **Youden's J:** maximise sensitivity + specificity − 1.
- **Target sensitivity:** the highest threshold that still catches ≥ 90% of positives on validation. Screening tasks often prefer this, because missing a case is costly.

### Exercise 19.2: `capstone/medlab/imaging/metrics.py`
```python
youden_threshold(y, p) -> float
threshold_for_sensitivity(y, p, target=0.9) -> float
binary_metrics(y, p, thr) -> {"sensitivity", "specificity", "ppv", "npv"}     # predict positive if p >= thr; 0/0 → 0.0
bootstrap_ci(stat_fn, y, p, n_boot=2000, seed=0) -> (low, high)              # stat_fn(y_sample, p_sample) -> float;
                                                                              # skip resamples with a single class
```
Add `tests/test_imaging_metrics.py`. Then build Table B: for each model, **average the test probabilities over the 3 seeds** (a small seed ensemble), and report test AUROC [CI], sensitivity and specificity at the Youden threshold from the (averaged) **validation** probabilities [CIs], and ECE.

In [ ]:
# Implement metrics.py and tests, then run the checker.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
assert run_tests("tests/test_imaging_metrics.py") == 0
im, calib, report = fresh("medlab.imaging.metrics"), fresh("medlab.imaging.calibration"), fresh("medlab.report")
y = np.array([0, 0, 1, 1]); p = np.array([0.1, 0.6, 0.4, 0.9])
assert im.binary_metrics(y, p, 0.5) == {"sensitivity": 0.5, "specificity": 0.5, "ppv": 0.5, "npv": 0.5}
assert im.threshold_for_sensitivity(y, p, 1.0) == 0.4

rows = []
for m in models:
    yv = np.array(res[(m, "val")]["labels"]); yt = np.array(res[(m, "test")]["labels"])
    pv = np.mean([s["probs"] for s in res[(m, "val")]["per_seed"]], axis=0)
    pt = np.mean([s["probs"] for s in res[(m, "test")]["per_seed"]], axis=0)
    thr = im.youden_threshold(yv, pv)
    bm = im.binary_metrics(yt, pt, thr)
    auc_ci = im.bootstrap_ci(roc_auc_score, yt, pt)
    sens_ci = im.bootstrap_ci(lambda a, b: im.binary_metrics(a, b, thr)["sensitivity"], yt, pt)
    spec_ci = im.bootstrap_ci(lambda a, b: im.binary_metrics(a, b, thr)["specificity"], yt, pt)
    rows.append({"model": m, "test AUROC": roc_auc_score(yt, pt), "AUROC CI": f"[{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]",
                 "threshold (val)": thr, "sensitivity": bm["sensitivity"], "sens CI": f"[{sens_ci[0]:.3f}, {sens_ci[1]:.3f}]",
                 "specificity": bm["specificity"], "spec CI": f"[{spec_ci[0]:.3f}, {spec_ci[1]:.3f}]",
                 "ECE": calib.expected_calibration_error(yt, pt)})
table_B = pd.DataFrame(rows).set_index("model")
print(table_B.round(3))
(CAP / "results" / "table_B.md").write_text(report.to_markdown(table_B))
print("✅ 19.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Youden: try every unique value of `p` as a threshold, compute sensitivity + specificity − 1, and keep the best.

</details>

<details><summary>Hint 2</summary>

Target sensitivity: sort the positives' scores in descending order; the threshold is the score at position `ceil(target * n_pos) - 1`.

</details>


✍️ *Which model would you choose for a *screening* setting, and at which threshold rule? What is the PPV at your threshold, and what would it be at a real-world prevalence of 5% instead of the dataset's ~60%? (Week 7: Bayes!)* (double-click to edit)



---
# Day 3: Robustness probe (2.5 h)

Models fail when the data shifts: a different scanner, different exposure, compression. A cheap probe is to perturb the **test** images synthetically and watch the AUROC fall. It doesn't prove robustness, but it can reveal fragility.

### Exercise 19.3: `capstone/medlab/imaging/robustness.py`
```python
perturb(X, kind, severity, seed=0) -> X'    # X uint8 (N, H, W); returns uint8, clipped to [0, 255]
#   "brightness": add severity * 50 to every pixel
#   "contrast":   scale (X - mean) by (1 - 0.3 * severity), then add the mean back
#   "noise":      add Gaussian noise with std severity * 15 (use the seed)
#   "blur":       a box blur with kernel size 2 * severity + 1 (scipy.ndimage.uniform_filter per image); severity 0 = unchanged
```
`severity` is in {0, 1, 2, 3}, and 0 must return the input unchanged. Add tests. Then retrain your **selected** model (one seed, CPU, or load a checkpoint) and plot test AUROC versus severity for each kind.

In [ ]:
# Implement robustness.py and tests, then run the checker.

In [ ]:
import matplotlib.pyplot as plt
assert run_tests("tests/test_robustness.py") == 0
rob, idata, imodels, itrain, data_mod = (fresh(m) for m in ["medlab.imaging.robustness", "medlab.imaging.data", "medlab.imaging.models", "medlab.imaging.train", "medlab.data"])
data = data_mod.load_medmnist("pneumoniamnist", size=28)
X0 = data["test"][0]
assert np.array_equal(rob.perturb(X0, "noise", 0), X0) and rob.perturb(X0, "brightness", 3).dtype == np.uint8
ds = idata.make_tensors(data); mean, std = ds["stats"]
to_t = lambda X: __import__("torch").tensor(((X.astype(np.float32) / 255 - mean) / std).astype(np.float32)).unsqueeze(1)
itrain.set_seed(0)
model, _ = itrain.fit(imodels.SmallCNN(), ds["train"], ds["val"].tensors[0], data["val"][1], epochs=3, seed=0)
curves = {}
for kind in ["brightness", "contrast", "noise", "blur"]:
    curves[kind] = [roc_auc_score(data["test"][1], itrain.predict_proba(model, to_t(rob.perturb(X0, kind, s)))) for s in range(4)]
    plt.plot(range(4), curves[kind], "o-", label=kind)
plt.xlabel("severity"); plt.ylabel("test AUROC"); plt.legend(); plt.title("Robustness probe (SmallCNN, seed 0)")
(CAP / "report" / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(CAP / "report" / "figures" / "fig_B_robustness.png", dpi=200, bbox_inches="tight"); plt.show()
json.dump(curves, open(CAP / "results" / "imaging" / "robustness.json", "w"), indent=2)
print("✅ 19.3 passed")

---
# Day 4: Figures and the Experiment B write-up (2 h)

### Exercise 19.4
1. `fig_B_roc.png`: test ROC curves for all models in one plot (`sklearn.metrics.roc_curve`), with AUROC in the legend.
2. `fig_B_reliability.png`: a reliability diagram for the selected model (Week 16).
3. `report/results_B.md` with the same structure as A: **Setup / Results / Clinical operating point / Robustness / Calibration / Deviations from protocol / Limitations**. The limitations must include: single-source data, low resolution, no patient-level split information, synthetic perturbations ≠ real shift, and **not clinically validated**.

In [ ]:
# Make the two figures and write results_B.md, then run the checker.

In [ ]:
for f in ["fig_B_roc.png", "fig_B_reliability.png", "fig_B_robustness.png"]:
    assert (CAP / "report" / "figures" / f).exists(), f
t = (CAP / "report" / "results_B.md").read_text().lower()
for h in ["setup", "results", "operating point", "robustness", "calibration", "deviations from protocol", "limitations", "not clinically validated"]:
    assert h in t, f"missing: {h}"
print("✅ 19.4 passed (milestone M8)")

---
# Review (30 min)
1. Why must the threshold come from validation?
2. How does prevalence change PPV but not sensitivity?
3. What can a synthetic robustness probe *not* tell you?
4. *(Week 8)* What's a paired bootstrap, and when would you use it here?
5. *(Week 14)* What does `model.eval()` change for a ResNet with batch norm?

✍️ *Your answer:* (double-click to edit)

